<a href="https://colab.research.google.com/github/anoopgadde/Reinforecement_Learning/blob/main/RL_LAB_EXAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# Reinforcement Learning Lab - Set 12
# State Value V(S0) and Action Value Q(S0, Up) in a 3x3 Grid World
# (No Bellman equation - paths are followed and rewards summed)
# ============================================================


In [7]:
# ---------- Task 1: State space and action space ----------
GRID = [
    ["S3", "X",  "S4"],
    ["S0", "S1", "S2"],
    ["S5", "S6", "G"],
]

OBSTACLE = "X"
GOAL = "G"
STATES = ["S0", "S1", "S2", "S3", "S4", "S5", "S6", "G"]

ACTIONS = {
    "Up":    (-1, 0),
    "Down":  (1, 0),
    "Left":  (0, -1),
    "Right": (0, 1),
}

# state name -> (row, col)
POS = {GRID[r][c]: (r, c) for r in range(3) for c in range(3)}

print("Grid World (3 x 3):")
for row in GRID:
    print(" | ".join(row))
print("State Space: {" + ", ".join(STATES) + "}")
print("Obstacle:", OBSTACLE)
print("Action Space: {" + ", ".join(ACTIONS.keys()) + "}")

Grid World (3 x 3):
S3 | X | S4
S0 | S1 | S2
S5 | S6 | G
State Space: {S0, S1, S2, S3, S4, S5, S6, G}
Obstacle: X
Action Space: {Up, Down, Left, Right}


In [8]:
# ---------- Task 2: Reward & transition functions ----------
NORMAL_REWARD = -1
GOAL_REWARD = 20
OBSTACLE_REWARD = -8   # given in the problem; a blocked attempt still gives -1
GAMMA = 0.9

def transition(state, action):
    """Deterministic transition -> (next_state, reward, terminated)"""
    r, c = POS[state]
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc

    # Outside grid: stay, -1
    if not (0 <= nr < 3 and 0 <= nc < 3):
        return state, NORMAL_REWARD, False

    next_state = GRID[nr][nc]

    # Obstacle: blocked, stay, -1
    if next_state == OBSTACLE:
        return state, NORMAL_REWARD, False

    # Goal: +20, episode ends
    if next_state == GOAL:
        return next_state, GOAL_REWARD, True

    # Normal move
    return next_state, NORMAL_REWARD, False

def fmt(x):
    return f"+{x}" if x > 0 else f"{x}"

print("Normal Movement Reward:", NORMAL_REWARD)
print("Goal Reward: +20")
print("Obstacle Reward: -8 (obstacle is blocked; attempted entry remains in current state)")
print("Discount Factor (gamma):", GAMMA)
print("Sample Transitions:")
for s, a in [("S0","Right"), ("S0","Up"), ("S1","Right"), ("S2","Down"), ("S3","Right")]:
    ns, rew, done = transition(s, a)
    note = " [blocked by X]" if (s, a) == ("S3", "Right") else ""
    print(f"T({s}, {a}) = {ns} | Reward = {fmt(rew)} | Terminated = {done}{note}")

Normal Movement Reward: -1
Goal Reward: +20
Obstacle Reward: -8 (obstacle is blocked; attempted entry remains in current state)
Discount Factor (gamma): 0.9
Sample Transitions:
T(S0, Right) = S1 | Reward = -1 | Terminated = False
T(S0, Up) = S3 | Reward = -1 | Terminated = False
T(S1, Right) = S2 | Reward = -1 | Terminated = False
T(S2, Down) = G | Reward = +20 | Terminated = True
T(S3, Right) = S3 | Reward = -1 | Terminated = False [blocked by X]


In [9]:
# ---------- Task 3: Policy ----------
POLICY = {
    "S0": "Right",
    "S1": "Right",
    "S2": "Down",
    "S3": "Down",
}

print("Policy:")
for s, a in POLICY.items():
    print(f"{s}: {a}")

Policy:
S0: Right
S1: Right
S2: Down
S3: Down


In [10]:
# ---------- Task 4: State value function ----------
def discounted_return(rewards, gamma=GAMMA):
    return sum((gamma ** t) * r for t, r in enumerate(rewards))

def follow_policy(start_state, max_steps=50):
    path, rewards, acts = [start_state], [], []
    state = start_state
    for _ in range(max_steps):
        if state not in POLICY:
            break
        action = POLICY[state]
        state, rew, done = transition(state, action)
        path.append(state)
        rewards.append(rew)
        acts.append(action)
        if done:
            break
    return path, rewards, acts

def state_value_function(start_state="S0"):
    path, rewards, _ = follow_policy(start_state)
    return path, rewards, discounted_return(rewards)

path, rewards, V = state_value_function("S0")
print("State Path:", " -> ".join(path))
print("Reward Sequence:", "[" + ", ".join(fmt(r) for r in rewards) + "]")
print("V(S0) = -1 + (0.9 x -1) + (0.9^2 x 20)")
print("V(S0) = -1 - 0.9 + 16.2")
print(f"State Value V(S0) = {V:.1f}")

State Path: S0 -> S1 -> S2 -> G
Reward Sequence: [-1, -1, +20]
V(S0) = -1 + (0.9 x -1) + (0.9^2 x 20)
V(S0) = -1 - 0.9 + 16.2
State Value V(S0) = 14.3


In [11]:
# ---------- Task 5: Action value function ----------
def action_value_function(state="S0", action="Up"):
    # Step 1: execute the given action
    next_state, r0, done = transition(state, action)
    path = [state, next_state]
    rewards = [r0]
    policy_actions = []

    # Step 2: follow the policy from the resulting state
    if not done:
        p, rs, acts = follow_policy(next_state)
        path += p[1:]
        rewards += rs
        policy_actions = acts

    return next_state, policy_actions, path, rewards, discounted_return(rewards)

ns, pol_acts, path, rewards, Q = action_value_function("S0", "Up")
print("Initial State: S0")
print("Selected Action: Up")
print("Next State:", ns)
print("Then follow policy:", " -> ".join(pol_acts))
print("State Path:", " -> ".join(path))
print("Reward Sequence:", "[" + ", ".join(fmt(r) for r in rewards) + "]")
print("Q(S0, Up) = -1 + (0.9 x -1) + (0.9^2 x -1) + (0.9^3 x -1) + (0.9^4 x 20)")
print("Q(S0, Up) = -1 - 0.9 - 0.81 - 0.729 + 13.122")
print(f"Action Value Q(S0, Up) = {Q:.3f}")

Initial State: S0
Selected Action: Up
Next State: S3
Then follow policy: Down -> Right -> Right -> Down
State Path: S0 -> S3 -> S0 -> S1 -> S2 -> G
Reward Sequence: [-1, -1, -1, -1, +20]
Q(S0, Up) = -1 + (0.9 x -1) + (0.9^2 x -1) + (0.9^3 x -1) + (0.9^4 x 20)
Q(S0, Up) = -1 - 0.9 - 0.81 - 0.729 + 13.122
Action Value Q(S0, Up) = 9.683
